<a href="https://colab.research.google.com/github/tskir/london-housing/blob/main/notebooks/date_field_null_rates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Planning London Datahub — date field null-rate check

Pulls a small subset of `applications` records (`valid_date` in 2021–2022, capped at 100,000 rows) and reports the % of rows where each date field is null/missing.


In [ ]:
import requests
import pandas as pd

pd.set_option('display.max_columns', 100)


## Config

In [ ]:
API_URL = "https://planningdata.london.gov.uk/api-guest"
INDEX = "applications"
HEADERS = {
    "X-API-AllowRequest": "be2rmRnt&",
    "Content-Type": "application/json",
}

PAGE_SIZE = 1000       # docs per scroll page
MAX_RECORDS = 100_000  # cap on the total number of rows pulled

FILTER_QUERY = {
    "range": {
        "valid_date": {
            "gte": "01/01/2021",
            "lte": "31/12/2022",
        }
    }
}

# Every date field we care about, including nested ones under application_details
DATE_FIELDS = [
    "actual_commencement_date",
    "actual_completion_date",
    "appeal_decision_date",
    "appeal_start_date",
    "application_details.existing_proposed_floorspace_details.actual_commencement_date",
    "application_details.existing_proposed_floorspace_details.actual_completion_date",
    "application_details.existing_proposed_floorspace_details.superseded_date",
    "application_details.intended_commencement_date",
    "application_details.intended_completion_date",
    "application_details.non_permanent_dwellings_details.actual_commencement_date",
    "application_details.non_permanent_dwellings_details.actual_completion_date",
    "application_details.non_permanent_dwellings_details.superseded_date",
    "application_details.open_and_protected_space_details.open_spaces_details.actual_commencement_date",
    "application_details.open_and_protected_space_details.open_spaces_details.actual_completion_date",
    "application_details.open_and_protected_space_details.open_spaces_details.superseded_date",
    "application_details.open_and_protected_space_details.protected_spaces_details.actual_commencement_date",
    "application_details.open_and_protected_space_details.protected_spaces_details.actual_completion_date",
    "application_details.open_and_protected_space_details.protected_spaces_details.superseded_date",
    "application_details.other_residential_accommodation_details.other_resi_accommodation_unit_details.actual_commencement_date",
    "application_details.other_residential_accommodation_details.other_resi_accommodation_unit_details.actual_completion_date",
    "application_details.other_residential_accommodation_details.other_resi_accommodation_unit_details.superseded_date",
    "application_details.phasing_details.intended_commencement_date",
    "application_details.phasing_details.intended_completion_date",
    "application_details.residential_details.residential_units.actual_commencement_date",
    "application_details.residential_details.residential_units.actual_completion_date",
    "application_details.residential_details.residential_units.superseded_date",
    "date_building_work_completed_under_previous_permission",
    "date_building_work_started_under_previous_permission",
    "decision_date",
    "decision_target_date",
    "lapsed_date",
    "last_date_consultation_comments",
    "last_synced",
    "last_updated",
    "pp_retry_expiry",
    "valid_date",
]


## Pull the data (scroll API)

Same scroll-based pull as the main exploration notebook, but scoped to `DATE_FIELDS` and capped at `MAX_RECORDS`.

In [ ]:
def fetch_subset(index, query, source_fields, page_size, max_records, scroll_ttl="2m"):
    all_hits = []

    init_body = {
        "size": page_size,
        "query": query,
        "_source": source_fields,
    }
    resp = requests.post(
        f"{API_URL}/{index}/_search?scroll={scroll_ttl}",
        headers=HEADERS,
        json=init_body,
    )
    resp.raise_for_status()
    data = resp.json()

    total = data["hits"]["total"]["value"] if isinstance(data["hits"]["total"], dict) else data["hits"]["total"]
    print(f"Matching records available: {total:,}")

    scroll_id = data.get("_scroll_id")
    hits = data["hits"]["hits"]

    while hits:
        all_hits.extend(hits)
        if len(all_hits) >= max_records:
            all_hits = all_hits[:max_records]
            break
        resp = requests.post(
            f"{API_URL}/_search/scroll",
            headers=HEADERS,
            json={"scroll": scroll_ttl, "scroll_id": scroll_id},
        )
        resp.raise_for_status()
        data = resp.json()
        scroll_id = data.get("_scroll_id")
        hits = data["hits"]["hits"]

    print(f"Pulled {len(all_hits):,} records")
    return all_hits


raw_hits = fetch_subset(INDEX, FILTER_QUERY, DATE_FIELDS, PAGE_SIZE, MAX_RECORDS)


## Flatten into a DataFrame

In [ ]:
sources = [h["_source"] for h in raw_hits]
df = pd.json_normalize(sources, sep=".")

print(df.shape)
df.head()


## % null by date field

Fields absent from every returned record (e.g. rarely-populated nested date fields) show as 100% null even if they don't appear as a column in `df` at all.

In [ ]:
n_rows = len(df)

null_pct = {}
for field in DATE_FIELDS:
    if field in df.columns:
        null_pct[field] = df[field].isna().mean() * 100
    else:
        null_pct[field] = 100.0

null_pct_df = (
    pd.Series(null_pct, name="pct_null")
    .sort_values(ascending=False)
    .to_frame()
)
null_pct_df["pct_null"] = null_pct_df["pct_null"].round(2)

pd.set_option('display.max_rows', None)
display(null_pct_df)
pd.reset_option('display.max_rows')


In [ ]:
ax = null_pct_df["pct_null"].plot(
    kind="barh", figsize=(10, 12), title=f"% null by date field (n={n_rows:,})"
)
ax.invert_yaxis()
ax.set_xlabel("% null")
ax.figure.tight_layout()
